In [1]:
"""
Project: "Deep-learning-based decomposition of overlapping-sparse images:
          application at the vertex of neutrino interactions"
Paper: https://arxiv.org/abs/2310.19695.
Author: Dr. Saul Alonso-Monsalve
Contact: salonso@ethz.ch/saul.alonso.monsalve@cern.ch
Description: Training script for the first decomposing transformer configuration.
"""

import os
import json
import torch
import pytorch_lightning as pl







In [4]:
from torch.utils.data import DataLoader
from datasets import VADataset
from datasets import TransformerConf3Dataset
from models import TransformerConf1, LightningModelTransformerConf1
from utils import args_transformer, SphericalAngularLoss
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar, EarlyStopping, Callback 



In [13]:
torch.set_float32_matmul_precision("medium")
pl_major = int(pl.__version__.split(".")[0])

class CustomProgressBar(TQDMProgressBar):
    def init_train_tqdm(self):
        bar = super().init_train_tqdm()
        bar.ascii = True  # Ensure ASCII characters are used
        
        return bar

    def init_validation_tqdm(self):
        bar = super().init_validation_tqdm()

        bar.ascii = True  # Ensure ASCII characters are used for validation
    
        return bar




In [16]:
class PrintEpochMetrics(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        ep = int(trainer.current_epoch)
        tl = float(m.get("train_loss", float("nan")))
        vl = float(m.get("val_loss", float("nan")))
        lr = float(m.get("lr", pl_module.optimizers().param_groups[0]['lr']))
        print(f"Epoch {ep:03d} | train_loss={tl:.4f} | val_loss={vl:.4f} | lr={lr:.3g}")

In [21]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [24]:

torch.multiprocessing.set_sharing_strategy('file_system')
parser = args_transformer(1)
args, unknown = parser.parse_known_args()
print("\n- Arguments:")
#for arg, value in vars(args).items():
#    print(f"  {arg}: {value}")
nb_gpus = len(args.gpus)
gpus = ', '.join(args.gpus) if nb_gpus > 1 else str(args.gpus[0])


# Manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = gpus

args.metadata_path = "/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Data/Highland_Output/NN_Data/metadata.pkl"
args.dataset_path = "/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Data/Highland_Output/NN_Data/{}/{}/{}.npz"
args.save_dir = "/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Codes/NeutrinoVertex-DL-sfgd_develop/NeutrinoVertex-DL/train/"
args.checkpoint_path = "/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Codes/NeutrinoVertex-DL-sfgd_develop/NeutrinoVertex-DL/train/checkpoints/"
args.checkpoint_name = "v1"
args.epochs = 200
args.log_every_n_steps = 50

# Training and validation sets
train_set = VADataset(args, split="train")
print("train_set length: ", len(train_set))
val_set = VADataset(args, split="val")
print("val_set length: ", len(val_set))
# Training and validation loaders
train_loader = DataLoader(train_set, batch_size=args.batch_size, num_workers=args.num_workers,
                          collate_fn=train_set.collate_fn, pin_memory=True, persistent_workers=True, shuffle=True)
val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.num_workers,
                        collate_fn=val_set.collate_fn, pin_memory=True, persistent_workers=True, shuffle=False)
    
# Initialise model
model = TransformerConf1(num_encoder_layers=args.encoder_layers,
                         num_decoder_layers=args.decoder_layers,
                         emb_size=args.hidden,
                         num_head=args.attn_heads,
                         img_size=args.va_size,
                         dropout=args.dropout,
                         max_len=5,
                         )
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable params: {}".format(total_params))

# Loss functions
loss_fn1 = torch.nn.MSELoss()  # vertex position
loss_fn2 = torch.nn.MSELoss()  # ekin
loss_fn3 = SphericalAngularLoss()  # dirs
loss_fn4 = torch.nn.BCEWithLogitsLoss()  # keep iterating

    
# Calculate arguments for scheduler
nb_batches = len(train_loader)
denom = args.accum_grad_batches * nb_gpus
print(nb_batches)
print(denom)

    
#args.lr = args.lr * (args.batch_size * denom) / 256.
args.scheduler_steps = nb_batches * args.cosine_annealing_steps // denom
args.warmup_steps = nb_batches * args.warmup_steps // denom
args.start_cosine_step = (nb_batches * args.epochs // denom) - args.scheduler_steps
print(f"lr                = {args.lr}")
print(f"scheduler_steps   = {args.scheduler_steps}")
print(f"warmup_steps      = {args.warmup_steps}")
print(f"start_cosine_step = {args.start_cosine_step}")
print(f"eff. batch size   = {args.batch_size * denom}")


# Define logger and checkpoint
logger = CSVLogger(save_dir=args.save_dir + "/logs", name=args.name)
tb_logger = TensorBoardLogger(save_dir=args.save_dir + "/tb_logs", name=args.name)
callbacks = []
monitored_losses = ['val_loss',]
    
for loss_name in monitored_losses:
    checkpoint = ModelCheckpoint(
        dirpath=f"{args.checkpoint_path}/{args.checkpoint_name}/{loss_name}",
        save_top_k=args.save_top_k,
        monitor=loss_name,
        mode="min",
        save_last=True
    )
    callbacks.append(checkpoint)

progress_bar = CustomProgressBar()
callbacks.append(progress_bar)
if args.early_stop_patience > 0:
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=args.early_stop_patience,
        verbose=True,
        mode='min' 
    )
    callbacks.append(early_stop_callback)

    

# Create lightning model
lightning_model = LightningModelTransformerConf1(model=model,
                                                 loss_fn1=loss_fn1,
                                                 loss_fn2=loss_fn2,
                                                 loss_fn3=loss_fn3,
                                                 loss_fn4=loss_fn4,
                                                 args=args,
                                                 )
callbacks.append(PrintEpochMetrics())
    

# Log the hyperparameters
logger.log_hyperparams(vars(args))
tb_logger.log_hyperparams(vars(args))

# Create trainer module
trainer = pl.Trainer(
    max_epochs=args.epochs,
    callbacks=callbacks,
    accelerator="gpu",
    precision="bf16-mixed" if pl_major >= 2 else 32,
    devices=nb_gpus,
    strategy="ddp" if nb_gpus > 1 else "auto",
    logger=[logger, tb_logger],
    log_every_n_steps=args.log_every_n_steps,
    deterministic=True,
    accumulate_grad_batches=args.accum_grad_batches,
)

    

# Run the training
trainer.fit(
    model=lightning_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=args.load_checkpoint if args.load_checkpoint else None,
)



/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Environment/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)



- Arguments:
train_set length:  8
val_set length:  8
TransformerConf1(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
          )
          (linear1): Linear(in_features=16, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=64, out_features=16, bias=True)
          (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
        (0): TransformerDecoderLayer(
       

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Environment/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:701: Checkpoint directory /mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Codes/NeutrinoVertex-DL-sfgd_develop/NeutrinoVertex-DL/train/checkpoints/v1/val_loss exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Environment/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name     | Type                 | Params | Mode 
----------------------------------------------------------
0 | model    | TransformerConf1     | 8.5 K  | train
1 | loss_fn1 | MSELoss              | 0      | train
2 | loss_fn2 | MSEL

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Environment/lib/python3.12/site-packages/torch/nn/functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Environment/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 198 | train_loss=2.0295 | val_loss=2.3155 | lr=0.000191


Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199 | train_loss=2.8816 | val_loss=2.3146 | lr=4.89e-05


In [25]:
%reload_ext tensorboard
%tensorboard --logdir "/mnt/c/Users/btaol/Work/T2K_BL/Vertex_Activity/Codes/NeutrinoVertex-DL-sfgd_develop/NeutrinoVertex-DL/train/tb_logs/v1/tb_logs/v2/"